# Bài thực hành Deep Learning: Artificial Neural Network

## Mục tiêu bài thực hành

- Cài đặt ANN cho 5 bài toán: CIFAR10, MNIST, Cat/Dog, Adult Income và Car Evaluation.
- Biết cách tiền xử lý dữ liệu ảnh bằng chuẩn hóa pixel và flatten ảnh thành vector 1 chiều.
- Biết cách tiền xử lý dữ liệu bảng bằng missing value, one-hot encoding và scaling.
- Train, đánh giá, vẽ biểu đồ accuracy/loss và lưu model vào thư mục `models/`.

## Giới thiệu ANN

Artificial Neural Network là mạng neural gồm các tầng neuron kết nối với nhau. Trong bài này ta dùng `Sequential`, các tầng `Dense`, activation `relu` cho hidden layer, `Dropout` để giảm overfitting. Bài phân loại nhiều lớp dùng `softmax`, bài nhị phân dùng `sigmoid`. Vì đây là bài ANN nên không dùng `Conv2D` hoặc `MaxPooling2D`.

## Import thư viện

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.datasets import cifar10, mnist

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

Path('models').mkdir(exist_ok=True)
Path('static/training_plots').mkdir(parents=True, exist_ok=True)

def plot_history(history, title):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history.get('accuracy', []), label='train')
    plt.plot(history.history.get('val_accuracy', []), label='val')
    plt.title(title + ' accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history.get('loss', []), label='train')
    plt.plot(history.history.get('val_loss', []), label='val')
    plt.title(title + ' loss')
    plt.legend()
    plt.show()

## Bài 1: ANN CIFAR10

CIFAR10 gồm 10 lớp: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. Ảnh gốc có kích thước 32x32x3, sau khi flatten có 3072 chiều.

In [ ]:
cifar_labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()

plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_cifar[i])
    plt.title(cifar_labels[int(y_train_cifar[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

x_train_cifar_flat = x_train_cifar.astype('float32').reshape(len(x_train_cifar), -1) / 255.0
x_test_cifar_flat = x_test_cifar.astype('float32').reshape(len(x_test_cifar), -1) / 255.0

model_cifar = Sequential([
    Input(shape=(3072,)),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])
model_cifar.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_cifar.summary()

# Bỏ comment 2 dòng dưới để train trong notebook.
# history_cifar = model_cifar.fit(x_train_cifar_flat, y_train_cifar, epochs=10, batch_size=128, validation_split=0.1)
# plot_history(history_cifar, 'CIFAR10 ANN')

## Bài 2: ANN MNIST

MNIST gồm ảnh chữ số 0 đến 9. Ảnh gốc 28x28 grayscale, sau khi flatten có 784 chiều.

In [ ]:
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = mnist.load_data()

plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_mnist[i], cmap='gray')
    plt.title(str(y_train_mnist[i]))
    plt.axis('off')
plt.tight_layout()
plt.show()

x_train_mnist_flat = x_train_mnist.astype('float32').reshape(len(x_train_mnist), -1) / 255.0
x_test_mnist_flat = x_test_mnist.astype('float32').reshape(len(x_test_mnist), -1) / 255.0

model_mnist = Sequential([
    Input(shape=(784,)),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])
model_mnist.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_mnist.summary()

# history_mnist = model_mnist.fit(x_train_mnist_flat, y_train_mnist, epochs=10, batch_size=128, validation_split=0.1)
# plot_history(history_mnist, 'MNIST ANN')

## Bài 3: ANN Cat/Dog

Dữ liệu ưu tiên dùng TensorFlow Datasets `cats_vs_dogs`. Ảnh được resize về 64x64x3, chuẩn hóa và flatten thành vector 12288 chiều. Output dùng 1 neuron sigmoid.

In [ ]:
# Cell này tải một vài ảnh mẫu. Nếu lỗi mạng, có thể dùng dữ liệu local theo README.
try:
    import tensorflow_datasets as tfds
    sample_ds = tfds.load('cats_vs_dogs', split='train[:8]', as_supervised=True)
    plt.figure(figsize=(8, 4))
    for i, (image, label) in enumerate(sample_ds):
        plt.subplot(2, 4, i + 1)
        plt.imshow(image.numpy())
        plt.title('dog' if int(label.numpy()) == 1 else 'cat')
        plt.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Không tải được cats_vs_dogs:', e)

model_catdog = Sequential([
    Input(shape=(64 * 64 * 3,)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_catdog.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_catdog.summary()

# Để train đầy đủ, chạy: python training/train_catdog_ann.py

## Bài 4: ANN Adult Income

Bài toán phân loại nhị phân: `<=50K` hoặc `>50K`. Dữ liệu gồm cột numeric và categorical, cần impute missing value, one-hot encoding và StandardScaler.

In [ ]:
adult_columns = ['age','workclass','fnlwgt','education','education-num','marital-status','occupation','relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income']
try:
    adult_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data', names=adult_columns, na_values=[' ?', '?'], skipinitialspace=True)
    display(adult_df.head())
except Exception as e:
    print('Không tải được Adult:', e)

numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

model_adult = Sequential([
    Input(shape=(1,)),  # Khi train thật, input_dim bằng số cột sau one-hot.
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_adult.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Để train đầy đủ và lưu preprocessor, chạy: python training/train_adult_ann.py

## Bài 5: ANN Car Evaluation

Input gồm `buying`, `maint`, `doors`, `persons`, `lug_boot`, `safety`. Output gồm 4 lớp: `unacc`, `acc`, `good`, `vgood`.

In [ ]:
car_columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
try:
    car_df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data', names=car_columns)
    display(car_df.head())
except Exception as e:
    print('Không tải được Car Evaluation:', e)

model_car = Sequential([
    Input(shape=(1,)),  # Khi train thật, input_dim bằng số cột sau one-hot.
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')
])
model_car.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Để train đầy đủ và lưu encoder, chạy: python training/train_car_ann.py

## Tổng kết kết quả

## Chạy train đầy đủ và hiển thị biểu đồ

Các cell phía trên mô tả kiến trúc và hiển thị dữ liệu mẫu. Để train đầy đủ, chạy các script sau. Sau khi train, biểu đồ accuracy/loss sẽ được lưu trong `static/training_plots/`.

In [ ]:
# Bỏ comment từng dòng để train model mong muốn.
# !python training/train_cifar10_ann.py
# !python training/train_mnist_ann.py
# !python training/train_catdog_ann.py
# !python training/train_adult_ann.py
# !python training/train_car_ann.py

In [ ]:
from IPython.display import Image, display

plot_files = [
    'static/training_plots/cifar10_history.png',
    'static/training_plots/mnist_history.png',
    'static/training_plots/catdog_history.png',
    'static/training_plots/adult_history.png',
    'static/training_plots/car_history.png',
]

for plot_file in plot_files:
    if Path(plot_file).exists():
        print(plot_file)
        display(Image(filename=plot_file))
    else:
        print('Chưa có biểu đồ:', plot_file)

In [ ]:
summary = pd.DataFrame([
    ['CIFAR10', 'TensorFlow/Keras CIFAR10', '3072', 'Dense 512, Dropout, Dense 256, Dropout, Dense 10', 'sparse_categorical_crossentropy', 'models/ann_cifar10.h5'],
    ['MNIST', 'TensorFlow/Keras MNIST', '784', 'Dense 256, Dropout, Dense 128, Dense 10', 'sparse_categorical_crossentropy', 'models/ann_mnist.h5'],
    ['Cat/Dog', 'TensorFlow Datasets cats_vs_dogs', '12288', 'Dense 256, Dropout, Dense 128, Dropout, Dense 1', 'binary_crossentropy', 'models/ann_catdog.h5'],
    ['Adult Income', 'UCI Adult', 'sau one-hot', 'Dense 128, Dropout, Dense 64, Dense 1', 'binary_crossentropy', 'models/ann_adult.h5'],
    ['Car Evaluation', 'UCI Car Evaluation', 'sau one-hot', 'Dense 64, Dropout, Dense 32, Dense 4', 'sparse_categorical_crossentropy', 'models/ann_car.h5'],
], columns=['Tên bài', 'Dataset', 'Input shape', 'Số lớp', 'Loss function', 'File model lưu'])
display(summary)

## Nhận xét

- ANN hoạt động tốt với dữ liệu bảng sau khi one-hot encoding và scaling phù hợp.
- Với ảnh, ANN vẫn có thể phân loại sau khi flatten, nhưng mất thông tin không gian nên thường không mạnh bằng CNN.
- Dropout giúp giảm overfitting khi mô hình có nhiều tham số.
- Cần kiểm tra accuracy/loss trên validation hoặc test để đánh giá mô hình khách quan.

## Kết luận

Bài thực hành hoàn thành quy trình xây dựng ANN: tải dữ liệu, tiền xử lý, thiết kế mô hình Dense, train, đánh giá, lưu model và triển khai dự đoán bằng Flask. Để nộp bài, nên chạy các script trong thư mục `training/` để sinh đầy đủ file model và biểu đồ.